In [68]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u


i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Desktop/comparision/Not_Source.txt"


colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")


ZP = {
    "i": (30.8603, 0.0024),
    "z": (30.4519, 0.0036),
    "y": (28.2513, 0.0029)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)

# ============================================================
# FORCE I-band non-detections to limiting magnitude
# ============================================================
I_LIMIT = 27.46

i_nondet = (~i_detected) | (i_snr < 2.0)

i_mag[i_nondet] = I_LIMIT
i_err[i_nondet] = 0.0


# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 4.0
cut_y_snr      = magerr_to_snr(y_err) > 4.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

cut_i_dropout  = i_nondet
cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After SNR-only cuts: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")

lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal candidates:")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f}, "
        f"I_mag={row.I_mag:.2f}, z_mag={row.Z_mag:.2f}, y_mag={row.Y_mag:.2f}"
    )


Y catalog size: 414747
After SNR-only cuts: 581 candidates
Final LBG candidates after red-list removal: 134
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat

Final candidates:
RA=357.278631, DEC=-31.660724, I_SNR=0.00, I_mag=27.46, z_mag=25.42, y_mag=23.89
RA=357.430310, DEC=-31.651233, I_SNR=1.98, I_mag=27.46, z_mag=25.55, y_mag=24.62
RA=356.766797, DEC=-31.636807, I_SNR=0.64, I_mag=27.46, z_mag=25.43, y_mag=24.44
RA=356.715290, DEC=-31.636123, I_SNR=0.00, I_mag=27.46, z_mag=25.55, y_mag=24.52
RA=357.459845, DEC=-31.629389, I_SNR=0.00, I_mag=27.46, z_mag=25.54, y_mag=24.72
RA=356.449227, DEC=-31.624836, I_SNR=0.92, I_mag=27.46, z_mag=25.41, y_mag=24.50
RA=356.946582, DEC=-31.611597, I_SNR=0.00, I_mag=27.46, z_mag=25.72, y_mag=24.72
RA=356.860591, DEC=-31.590200, I_SNR=1.87, I_mag=27.46, z_mag=24.11, y_mag=23.26
RA=357.184588, DEC=-31.587619, I_SNR=0.01, I_mag=27.46, z_mag=24.86, y_mag=23.86
RA=357.260189, DEC=-31.580650, I_SNR=0.71, I_mag=27.46, z_mag=25.72, y_mag

In [6]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Desktop/comparision/Not_Source_updated.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.8603, 0.0024),
    "z": (30.4519, 0.0036),
    "y": (28.2513, 0.0029)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# I-band non-detections ONLY
# ============================================================
I_LIMIT = 27.46

i_nondet = (~i_detected) | (~np.isfinite(i_mag))

# apply limit ONLY to true non-detections
i_mag_lim = i_mag.copy()
i_err_lim = i_err.copy()

i_mag_lim[i_nondet] = I_LIMIT
i_err_lim[i_nondet] = np.inf   # important

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag_lim - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 4.0
cut_y_snr      = y_snr > 4.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

# dropout is LOGICAL, not photometric
cut_i_dropout  = (~i_detected) | (i_snr < 2.0)

cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"Selected LBG candidates: {len(lbg_idx)}")

# ============================================================
# Build output catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag_lim[lbg_idx],
    "I_err": i_err_lim[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")

lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal candidates:")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f}, "
        f"I_mag={row.I_mag:.2f}, "
        f"Z_mag={row.Z_mag:.2f}, "
        f"Y_mag={row.Y_mag:.2f}"
    )


Y catalog size: 414747
Selected LBG candidates: 577
Final LBG candidates: 135
Saved to /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat

Final candidates:
RA=357.278631, DEC=-31.660724, I_SNR=0.00, I_mag=27.46, Z_mag=25.42, Y_mag=23.89
RA=357.430310, DEC=-31.651233, I_SNR=1.98, I_mag=26.67, Z_mag=25.55, Y_mag=24.62
RA=356.766797, DEC=-31.636807, I_SNR=0.64, I_mag=27.88, Z_mag=25.43, Y_mag=24.44
RA=356.715290, DEC=-31.636123, I_SNR=0.00, I_mag=27.46, Z_mag=25.55, Y_mag=24.52
RA=357.459845, DEC=-31.629389, I_SNR=0.00, I_mag=27.46, Z_mag=25.54, Y_mag=24.72
RA=356.449227, DEC=-31.624836, I_SNR=0.92, I_mag=27.50, Z_mag=25.41, Y_mag=24.50
RA=356.946582, DEC=-31.611597, I_SNR=0.00, I_mag=27.46, Z_mag=25.72, Y_mag=24.72
RA=356.860591, DEC=-31.590200, I_SNR=1.87, I_mag=26.70, Z_mag=24.11, Y_mag=23.26
RA=357.184588, DEC=-31.587619, I_SNR=0.01, I_mag=129.86, Z_mag=24.86, Y_mag=23.86
RA=356.837914, DEC=-31.585759, I_SNR=0.00, I_mag=27.46, Z_mag=25.55, Y_mag=24.61
RA=357.260189, DEC=-31

In [3]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_I_final.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Z_final.cat"
y_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_final.cat"

redlist_file = "/Users/aishwarya/Desktop/comparision/Not_Source.txt"

# ============================================================
# Load catalogs
# ============================================================
colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zero points
# ============================================================
ZP = {
    "i": (30.8603, 0.0024),
    "z": (30.4519, 0.0036),
    "y": (28.2513, 0.0029)
}

# ============================================================
# Calibrate magnitudes
# ============================================================
def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# I-band non-detections ONLY
# ============================================================
I_LIMIT = 27.46

i_nondet = (~i_detected) | (~np.isfinite(i_mag))

# apply limit ONLY to true non-detections
i_mag_lim = i_mag.copy()
i_err_lim = i_err.copy()

i_mag_lim[i_nondet] = I_LIMIT
i_err_lim[i_nondet] = np.inf   # important

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag_lim - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 3.0
cut_y_snr      = y_snr > 3.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

# dropout is LOGICAL, not photometric
cut_i_dropout  = (~i_detected) | (i_snr < 2.0)

cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"Selected LBG candidates: {len(lbg_idx)}")

# ============================================================
# Build output catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag_lim[lbg_idx],
    "I_err": i_err_lim[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")

lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/Final_LBG_candidates_trial.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal candidates:")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f}, "
        f"I_mag={row.I_mag:.2f}, "
        f"Z_mag={row.Z_mag:.2f}, "
        f"Y_mag={row.Y_mag:.2f}"
    )


Y catalog size: 414747
Selected LBG candidates: 804
Final LBG candidates: 358
Saved to /Users/aishwarya/Documents/Lyman_alpha/Final_LBG_candidates_trial.cat

Final candidates:
RA=357.366106, DEC=-31.803211, I_SNR=0.00, I_mag=27.46, Z_mag=25.79, Y_mag=24.67
RA=357.317260, DEC=-31.801071, I_SNR=0.00, I_mag=27.46, Z_mag=25.87, Y_mag=24.75
RA=356.708964, DEC=-31.785509, I_SNR=0.00, I_mag=27.46, Z_mag=25.78, Y_mag=24.58
RA=357.488459, DEC=-31.760797, I_SNR=0.00, I_mag=27.46, Z_mag=25.80, Y_mag=24.78
RA=356.746755, DEC=-31.740070, I_SNR=0.00, I_mag=27.46, Z_mag=25.78, Y_mag=24.40
RA=357.281993, DEC=-31.662408, I_SNR=0.00, I_mag=27.46, Z_mag=25.75, Y_mag=24.33
RA=357.291792, DEC=-31.662634, I_SNR=0.00, I_mag=27.46, Z_mag=26.02, Y_mag=24.21
RA=357.260512, DEC=-31.662100, I_SNR=0.00, I_mag=27.46, Z_mag=25.81, Y_mag=24.51
RA=357.247445, DEC=-31.661039, I_SNR=0.00, I_mag=27.46, Z_mag=25.69, Y_mag=24.73
RA=357.296180, DEC=-31.660890, I_SNR=0.00, I_mag=27.46, Z_mag=25.80, Y_mag=24.60
RA=357.268377,

In [63]:
import pandas as pd
import os


def cat_to_reg(cat_file,
               ra_col="RA",
               dec_col="DEC",
               point_style="circle",
               point_size=6,
               color="cyan",
               radius_arcsec=1.0):
    """
    Convert a catalog to DS9 region files:
    - point markers
    - 1 arcsec radius circles
    (NO labels)
    """

    base = cat_file.replace(".cat", "")
    reg_point = base + "_points.reg"
    reg_circle = base + "_r1arcsec.reg"

    df = pd.read_csv(cat_file)

    # ---------- POINT REGIONS ----------
    with open(reg_point, "w", encoding="utf8") as f:
        f.write("# Region file format: DS9 version 4.1\n")
        f.write(f"global color={color} width=2\n")
        f.write("fk5\n")

        for _, row in df.iterrows():
            f.write(
                f"point({row[ra_col]},{row[dec_col]}) "
                f"# point={point_style} {point_size}\n"
            )

    # ---------- 1 ARCSEC CIRCLES ----------
    with open(reg_circle, "w", encoding="utf8") as f:
        f.write("# Region file format: DS9 version 4.1\n")
        f.write(f"global color={color} width=2\n")
        f.write("fk5\n")

        for _, row in df.iterrows():
            f.write(
                f'circle({row[ra_col]},{row[dec_col]},{radius_arcsec}")\n'
            )

    print(f"Created:")
    print(f"  {reg_point}")
    print(f"  {reg_circle}")


if __name__ == "__main__":

    CAT_FILES = [
        "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
    ]

    for cat in CAT_FILES:
        if not os.path.exists(cat):
            print(f"File not found: {cat}")
            continue

        cat_to_reg(
            cat_file=cat,
            ra_col="RA",
            dec_col="DEC",
            point_style="circle",
            point_size=6,
            color="cyan",
            radius_arcsec=1.0
        )

    print("All catalogs converted to DS9 region files.")


Created:
  /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY_points.reg
  /Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY_r1arcsec.reg
All catalogs converted to DS9 region files.


In [66]:
import pandas as pd

in_path = "/Users/aishwarya/Desktop/Not_Source.csv"   # exported from Numbers
out_path = "/Users/aishwarya/Desktop/Not_Source.txt"

df = pd.read_csv(in_path, sep=";", header=None)

# keep only the first two numeric columns
ra = df.iloc[:, 1]
dec = df.iloc[:, 2]

clean = pd.DataFrame({"RA": ra, "DEC": dec}).dropna()

clean.to_csv(out_path, sep=" ", header=False, index=False)
print(f"Saved to {out_path}")


Saved to /Users/aishwarya/Desktop/Not_Source.txt


In [29]:
file_path = "/Users/aishwarya/Desktop/Not_Source.reg"

with open(file_path, "r") as f:
    n_rows = sum(1 for line in f if line.strip() and not line.startswith("#"))

print(f"Number of rows: {n_rows}")


Number of rows: 272


In [ ]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
lbg_file = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY.cat"
not_file = "/Users/aishwarya/Desktop/Not_Source.txt"
out_file = "/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY_cleaned.cat"

match_radius = 1.0 * u.arcsec

# ============================================================
# Load LBG catalog (CSV)
# ============================================================
lbg_df = pd.read_csv(lbg_file)

print(f"LBG catalog size (input): {len(lbg_df)}")

lbg_coords = SkyCoord(
    ra=lbg_df["RA"].values * u.deg,
    dec=lbg_df["DEC"].values * u.deg,
    frame="icrs"
)

# ============================================================
# Load Not_Source list (RA DEC)
# ============================================================
not_df = pd.read_csv(
    not_file,
    delim_whitespace=True,
    header=None,
    names=["RA", "DEC"]
)

# Fix DEC sign if needed (force negative)
# Convert to numeric first
not_df["DEC"] = pd.to_numeric(not_df["DEC"], errors="coerce")

# Now force negative
not_df["DEC"] = -np.abs(not_df["DEC"])


print(f"Not_Source list size: {len(not_df)}")

not_coords = SkyCoord(
    ra=not_df["RA"].values * u.deg,
    dec=not_df["DEC"].values * u.deg,
    frame="icrs"
)

# ============================================================
# Match and remove
# ============================================================
idx, sep, _ = lbg_coords.match_to_catalog_sky(not_coords)
remove_mask = sep < match_radius

print(f"Sources to remove: {remove_mask.sum()}")

clean_df = lbg_df.loc[~remove_mask].copy()

# ============================================================
# Save cleaned catalog
# ============================================================
clean_df.to_csv(out_file, index=False)

print("======================================")
print(f"Cleaned catalog saved to:\n{out_file}")
print(f"Final catalog size: {len(clean_df)}")
print("======================================")


LBG catalog size (input): 173
Not_Source list size: 435
Sources to remove: 8
Cleaned catalog saved to:
/Users/aishwarya/Documents/Lyman_alpha/LBG_candidates_ZY_cleaned.cat
Final catalog size: 165


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_8895/1395965103.py:34: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  not_df = pd.read_csv(


In [54]:
print(not_df.head())
print(not_df.describe())


           RA        DEC
0  357.982143 -31.144592
1  357.697380 -31.143542
2  358.069094 -30.825562
3  358.086406 -30.821694
4  357.852344 -30.702952
               RA         DEC
count  428.000000  428.000000
mean   357.075554  -31.600492
std      0.559536   15.785450
min    356.052356 -357.220314
25%    356.571657  -31.312197
50%    357.124280  -30.828309
75%    357.516176  -30.360794
max    358.266001  -29.840936


In [37]:
import pandas as pd

not_file = "/Users/aishwarya/Desktop/Not_Source.txt"

not_df = pd.read_csv(
    not_file,
    delim_whitespace=True,
    header=None,
    names=["RA", "DEC"]
)

print(not_df.head())
print(not_df.describe())

# Check invalid DEC values
bad_dec = not_df[(not_df["DEC"] < -90) | (not_df["DEC"] > 90)]

print(f"Bad DEC entries: {len(bad_dec)}")
print(bad_dec.head(10))


           RA        DEC
0  357.982143 -31.144592
1  357.697380 -31.143542
2  358.069094 -30.825562
3  358.086406 -30.821694
4  357.852344 -30.702952
               RA         DEC
count  428.000000  428.000000
mean   357.075554  -29.931239
std      0.559536   18.765209
min    356.052356  -31.804153
25%    356.571657  -31.308720
50%    357.124280  -30.827264
75%    357.516176  -30.354376
max    358.266001  357.220314
Bad DEC entries: 1
             RA         DEC
320  357.220314  357.220314


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_8895/94968728.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  not_df = pd.read_csv(
